# 부록 그래프·표 만들기 — 원래 V4(취약 후보 43개) 기준

발표그래프 노트북과 같은 방식으로 재현 코드(02·03·04)를 원래 V4로 다시 돌린 뒤, **부록 A1~A11에 넣을 그래프와 표**를 `results/부록그래프/`에 PNG와 엑셀로 저장합니다.

| 파일 | 부록 |
|---|---|
| `A1_정제_흐름.png` | A1 데이터 정제 (카드 원본에서 직접 계산) |
| `A2_변수표.png`, `A2_월별_편향.png`, `A2_baseline_설정.png` | A2 평소 매출 모델 |
| `A3_이벤트_달력.png`, `A3_이벤트표.png` | A3 이벤트·특수일 |
| `A4_전체판정.png`, `A4_가짜사건점검.png` | A4 취약 판정 |
| `A5_3축규칙.png` | A5 3축 계산 규칙 |
| `A6_유형기준.png`, `A6_성격x시간대.png`, `A6_업종성격_키워드.png` | A6 유형 분류 |
| `A7_후보_강남폭염.png` 외 3장 | A7 취약 후보 43개 |
| `A8_방문지출_주vs여름.png`, `A8_여름기준_바뀐업종.png`, `A8_외지인_거주민.png` | A8 감소 분해 |
| `A9_변수중요도.png`, `A9_검증묶음.png`, `A9_모델설정.png` | A9 2단계 모델 |
| `A10_과거평균비교_표.png`, `A10_후보찾기.png` | A10 과거 평균 비교 |
| `A11_GPU_CPU.png` (비교 결과가 있으면) | A11 재현성 |
| `부록표.xlsx` | 위 표 원본 |

**준비**: 발표그래프 노트북과 같습니다. `재현코드.zip`(또는 노트북 3개), 원래 V4(`outputs/38_최종baseline_V4.csv`), 카드 데이터1·2, 기상 파일. CPU 런타임, 약 5~10분.

A4의 가짜 사건 점검과 A5의 가중치 민감도는 재현 코드에 들어 있지 않은 이전 분석(`3축_재점검`) 결과라서 숫자를 표로만 넣습니다.


In [ ]:
# 0. 설정 — 경로 확인
from pathlib import Path
import io, json, time, contextlib, warnings
ROOT = Path('/content/drive/MyDrive/big_contest')
try:
    from google.colab import drive; drive.mount('/content/drive')
    import subprocess, sys; subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'koreanize-matplotlib', 'lightgbm'])
except ImportError:
    pass
import numpy as np, pandas as pd, matplotlib, matplotlib.pyplot as plt
import koreanize_matplotlib
warnings.filterwarnings('ignore')
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)

# ① 원래 V4 (43개가 나온 파일, 10/11 저장본). GPU로 01을 다시 돌린 파일이어도 됩니다.
V4_ORIG = ROOT / 'outputs' / '38_최종baseline_V4.csv'
# ② 결과 폴더 (기존 results/유형나누기 등은 건드리지 않음)
OUT = ROOT / 'results' / '부록그래프'; OUT.mkdir(parents=True, exist_ok=True)
EXPECT = {('강남(서울동남권)', '폭염'): 17, ('강남(서울동남권)', '호우'): 3, ('춘천', '폭염'): 14, ('춘천', '호우'): 9}

# 재현코드 노트북이 압축 파일(재현코드*.zip)로만 있으면 big_contest/재현코드_풀림 폴더에 풀어서 씀
import zipfile
if not list(ROOT.rglob('02_취약판정_유형나누기.ipynb')):
    zips = sorted(ROOT.rglob('재현코드*.zip'), key=lambda p: p.stat().st_mtime)
    assert zips, '재현코드 노트북이나 재현코드.zip을 Drive의 big_contest 아래에 올려 주세요'
    with zipfile.ZipFile(zips[-1]) as zf:
        for info in zf.infolist():
            name = info.filename
            if not info.flag_bits & 0x800:          # 한글 파일 이름 깨짐 방지
                try: name = name.encode('cp437').decode('utf-8')
                except UnicodeError: pass
            if name.endswith('.ipynb'):
                dst = ROOT / '재현코드_풀림' / Path(name).name; dst.parent.mkdir(exist_ok=True)
                dst.write_bytes(zf.read(info))
    print('압축 풀기:', zips[-1], '→', ROOT / '재현코드_풀림')

def find_nb(name):
    hits = sorted(ROOT.rglob(name), key=lambda p: p.stat().st_mtime)
    assert hits, f'{name}을(를) {ROOT} 아래에서 찾지 못했습니다 — 재현 코드 노트북을 Drive에 올려 주세요'
    return hits[-1]
NB02, NB03, NB04 = find_nb('02_취약판정_유형나누기.ipynb'), find_nb('03_2단계모델.ipynb'), find_nb('04_감소분해.ipynb')
assert V4_ORIG.exists(), f'V4_ORIG 경로를 확인하세요: {V4_ORIG}'
print('V4:', V4_ORIG, '| 수정 시각', time.strftime('%Y-%m-%d %H:%M', time.localtime(V4_ORIG.stat().st_mtime)))
print('노트북:', NB02, NB03, NB04, sep='\n  ')

# 발표용 그래프 스타일 (PPT 색에 맞춘 남색·주황 계열)
C = dict(navy='#1f2a5c', blue='#2a78d6', orange='#eb6834', teal='#1baf7a', yellow='#eda100', red='#d9534f',
         gray='#9aa0aa', ink='#16181d', ink2='#555a64', grid='#e6e7eb', shade='#fbe9e4', band='#d7efe9')
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': C['ink2'],
                     'axes.labelcolor': C['ink2'], 'xtick.color': C['ink2'], 'ytick.color': C['ink2'],
                     'axes.grid': True, 'grid.color': C['grid'], 'grid.linewidth': 0.8, 'axes.axisbelow': True,
                     'axes.unicode_minus': False, 'figure.dpi': 110, 'savefig.dpi': 220, 'font.size': 10})
def save(fig, name):
    fig.savefig(OUT / name, bbox_inches='tight', facecolor='white'); plt.show(); print('저장:', OUT / name)

# 표를 그림으로 저장 (PPT에 바로 붙이기용)
def table_png(df, name, title='', fs=9, col_w=None, note=''):
    df = df.copy().astype(object).where(df.notna(), '')
    n, m = df.shape
    fig, ax = plt.subplots(figsize=(min(1.6 * m, 16) if col_w is None else sum(col_w), 0.36 * (n + 1) + (0.5 if title else 0.1) + (0.35 if note else 0)))
    ax.axis('off')
    tb = ax.table(cellText=df.values.tolist(), colLabels=list(df.columns), loc='upper center', cellLoc='center',
                  colWidths=None if col_w is None else [w / sum(col_w) for w in col_w])
    tb.auto_set_font_size(False); tb.set_fontsize(fs); tb.scale(1, 1.35)
    for (r, c), cell in tb.get_celld().items():
        cell.set_edgecolor('#d9dbe1')
        if r == 0: cell.set_facecolor(C['navy']); cell.set_text_props(color='white', fontweight='bold')
        elif r % 2 == 0: cell.set_facecolor('#f4f5f8')
    if title: ax.set_title(title + (f'\n{note}' if note else ''), loc='left', fontsize=10.5 if not note else 10, color=C['ink'], pad=8)
    save(fig, name)
    return df
TABLES = {}

import matplotlib.dates as mdates


## 1. 재현 코드 실행 (02 → 03 → 04)

In [ ]:
# 1. 재현 코드 노트북을 불러 실행하는 함수 (경로만 바꾸고 출력은 숨김)
def run_nb(path, repl, stop=None, skip=()):
    nb = json.load(open(path, encoding='utf-8'))
    cells = [''.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code']
    ns = {'__name__': '__main__', '_CAP': {}}
    repl = {"ROOT = Path('/content/drive/MyDrive/big_contest')": f"ROOT = Path(r'{ROOT}')", **repl}
    done = {k: 0 for k in repl}
    log = io.StringIO()
    show, backend = plt.show, matplotlib.get_backend()
    plt.show = lambda *a, **k: plt.close('all')
    try:
        with contextlib.redirect_stdout(log):
            for i, src in enumerate(cells):
                if stop and src.startswith(stop): break
                if any(src.startswith(s) for s in skip): continue
                for old, new in repl.items():
                    if old in src: src = src.replace(old, new); done[old] += 1
                ns['display'] = lambda *a, **k: None
                exec(compile(src, f'{path.name}#cell{i}', 'exec'), ns)
                plt.close('all')
    finally:
        plt.show = show
    miss = [k for k, v in done.items() if v == 0]
    assert not miss, f'{path.name}에서 바꿀 줄을 찾지 못했습니다: {miss}'
    return ns, log.getvalue()

t0 = time.time()
ns2, log2 = run_nb(NB02, {
    "V4_PATH = find('38_최종baseline_V4.csv')": f"V4_PATH = Path(r'{V4_ORIG}')",
    "OUT = ROOT/'results'/'유형나누기'": f"OUT = Path(r'{OUT / '02_유형나누기'}')",
    "return tuple(np.percentile(ds / bs, RANGE_Q))": "_CAP[(g, tuple(dates))] = ds / bs; return tuple(np.percentile(ds / bs, RANGE_Q))",
}, stop='# 7.')
V4 = ns2['P']
TYPE_X = OUT / '02_유형나누기' / '유형나누기_결과.xlsx'
S2, CAND, EPI, EP2, D2 = ns2['S'], ns2['CAND'], ns2['EPI'], ns2['EP'], ns2['D']
cnt = CAND.groupby(['대상', '종류']).size().to_dict()
print(f'02 완료 ({time.time() - t0:.0f}초) — 취약 후보:', cnt, '합계', len(CAND))
assert cnt == EXPECT, '⚠ 43개가 아닙니다. V4_ORIG가 원래 V4(GPU) 파일인지 확인하세요 (CPU로 만든 V4면 47개)'


In [ ]:
# 2. 03(2단계 모델·과거 평균 비교), 04(감소 분해)
t0 = time.time()
ns3, log3 = run_nb(NB03, {
    "V4_PATH = find('38_최종baseline_V4.csv')": f"V4_PATH = Path(r'{V4_ORIG}')",
    "TYPE_PATH = find('유형나누기_결과.xlsx')": f"TYPE_PATH = Path(r'{TYPE_X}')",
    "OUT = ROOT/'results'/'2단계모델'": f"OUT = Path(r'{OUT / '03_2단계모델'}')",
}, skip=('# 10.',))
print(f'03 완료 ({time.time() - t0:.0f}초)')
t0 = time.time()
ns4, log4 = run_nb(NB04, {
    "V4_PATH = find('38_최종baseline_V4.csv')": f"V4_PATH = Path(r'{V4_ORIG}')",
    "TYPE_PATH = find('유형나누기_결과.xlsx')": f"TYPE_PATH = Path(r'{TYPE_X}')",
    "OUT = ROOT/'results'/'감소분해'": f"OUT = Path(r'{OUT / '04_감소분해'}')",
}, stop='# 6.')
print(f'04 완료 ({time.time() - t0:.0f}초)')
PE, MAE, AXV, REC, SUMM, IMP, CL, FEATS = ns3['PE'], ns3['MAE'], ns3['AXV'], ns3['REC'], ns3['SUMM'], ns3['imp'], ns3['CL'], ns3['FEATS']
T = ns4['T']
assert len(T) == 43


## A1. 데이터 정제

In [ ]:
# A1. 데이터 정제 흐름 — 카드 원본에서 직접 계산
def find_data(pat):
    hits = sorted(ROOT.rglob(pat)); assert hits, f'{pat}을(를) 찾지 못했습니다'; return hits[0]
raw = pd.read_csv(find_data('신한카드_빅콘테스트2026_데이터1*.txt'), sep='\t', encoding='cp949', dtype={'TA_YMD': str}, low_memory=False)
raw['TS_AT'] = pd.to_numeric(raw['TS_AT'], errors='coerce')
REG = {'서울 강남구': '강남', '강원 춘천시': '춘천'}
corp = raw['SEX_CCD'].eq('법인'); taxzz = raw['MCT_RY_CD'].isin(['세금공과금', 'ZZ_나머지'])
gn = raw['MCT_SGG_CD'].eq('서울 강남구')
keep = raw[~corp & ~taxzz]
cells_all = keep.groupby(['MCT_SGG_CD', 'MCT_RY_CD', 'TIME_GB']).ngroups
n_keep_cells = V4[['gkey', 'slot']].drop_duplicates().shape[0]
flow = pd.DataFrame([
    ['① 원본', f'{len(raw):,}행', f"{raw['TS_AT'].sum() / 1e8:,.0f}억 원", '데이터1 전체'],
    ['② 법인 제외', f'{(~corp).sum():,}행', f"{raw.loc[~corp, 'TS_AT'].sum() / 1e8:,.0f}억 원",
     f"법인: 행 {corp.mean()*100:.1f}% · 금액 {raw.loc[corp, 'TS_AT'].sum() / raw['TS_AT'].sum()*100:.1f}%"],
    ['③ 세금공과금·ZZ_나머지 제외', f'{len(keep):,}행', f"{keep['TS_AT'].sum() / 1e8:,.0f}억 원",
     f"두 업종이 강남 결제액의 {raw.loc[gn & taxzz, 'TS_AT'].sum() / raw.loc[gn, 'TS_AT'].sum()*100:.0f}%"],
    ['④ 지역×업종×시간대 집계', f'{cells_all:,}칸', '', '날짜×6시간 단위, 결제 없는 칸은 0'],
    ['⑤ 희소 칸 제외 (매출 있는 날 < 80%)', f'{n_keep_cells:,}칸', '', f"남은 지역×업종 {V4['gkey'].nunique()}개 (강남 {V4.loc[V4['대상'].str.startswith('강남'), 'gkey'].nunique()}, 춘천 {V4.loc[V4['대상'].eq('춘천'), 'gkey'].nunique()})"],
], columns=['단계', '규모', '금액', '내용'])
try:
    r2 = pd.read_csv(find_data('신한카드_빅콘테스트2026_데이터2*.txt'), sep='\t', encoding='cp949', usecols=['CLN_SGG_CD', 'TS_AT'], dtype=str)
    a2 = pd.to_numeric(r2['TS_AT'], errors='coerce')
    flow.loc[len(flow)] = ['(데이터2) 거주지 정보없음 제외', f"{(r2['CLN_SGG_CD'] == '정보없음').sum():,}행", '', f"매출의 {a2[r2['CLN_SGG_CD'] == '정보없음'].sum() / a2.sum()*100:.1f}%"]
except Exception as e:
    print('데이터2 건너뜀:', e)
del raw
TABLES['A1_정제흐름'] = table_png(flow, 'A1_정제_흐름.png', 'A1. 카드 데이터 정제 흐름', col_w=[3.2, 1.6, 1.6, 4.6])


## A2. 평소 매출 모델

In [ ]:
# A2. 평소 매출 모델 — 설정·변수표, 월별 편향
cfg = pd.DataFrame([
    ['모델', "XGBoost reg:tweedie (variance power 1.5), 목표 = 매출 / 1만 원"],
    ['하이퍼파라미터', '트리 1,000 · 학습률 0.05 · 깊이 8 · min_child_weight 5 · subsample 0.8 · seed 42'],
    ['학습 데이터', '평시만: 폭염·호우 특보 없음 & 일강수 < 10mm (한파·대설 날은 데이터에서 제외)'],
    ['교차 적합', 'ISO 주차 % 5로 5개 묶음, 예측하는 주 앞뒤 3일은 학습에서 제외'],
    ['공휴일', '요일=일요일·주말=1로 넣음: 강남 전 업종 + 춘천 11개 업종'],
    ['휴무 칸', '그 칸·요일의 평시 날 절반 이상이 매출 0 → 휴무(이후 분석에서 제외)'],
    ['사후 보정', '없음'],
    ['실행 환경', 'Colab T4 GPU · xgboost 3.4.1 (CPU로 돌리면 B가 칸 단위로 달라짐 → A11)'],
], columns=['항목', '설정'])
TABLES['A2_설정'] = table_png(cfg, 'A2_baseline_설정.png', 'A2. 평소 매출 모델(baseline V4) 설정', col_w=[2, 9], fs=9)
var = pd.DataFrame([
    ['범주 (3)', '지역 · 업종 · 시간대'],
    ['기본 달력 (3)', '요일 · 주말 · 공휴일'],
    ['달력 (8)', '월초 · 월말 · 추석직전 · 연휴위치 · 연휴전날 · 연휴다음날 · 연말 · 성탄이브'],
    ['변동성 (4)', '과거중앙값 · MAD비율 · 매출0비율 · 관측날수 (묶음마다 학습 평시 행으로만 계산)'],
    ['주기 달력 (4)', '일자 · 영업일순번 · 남은영업일 · 월내주차 (유치원 2곳에만)'],
    ['넣지 않은 것', '기온·강수 등 날씨 변수 (넣으면 날씨 피해까지 평소로 흡수됨)'],
], columns=['구분', '변수 (총 22개)'])
TABLES['A2_변수'] = table_png(var, 'A2_변수표.png', 'A2. baseline 변수 22개', col_w=[2, 9])

E = V4[V4['평시'] & ~V4['휴무'] & ~V4['특수일2']].copy()
E['월'] = E['날짜'].dt.month
mon = E.groupby(['대상', '월']).apply(lambda z: (z['S'] - z['B']).sum() / z.drop_duplicates(['gkey', '날짜'])['B_하루합'].sum() * 100).unstack(0)
days = E.drop_duplicates(['대상', '날짜']).groupby(['대상', '월']).size().unstack(0)
fig, ax = plt.subplots(figsize=(8, 3.8)); x = np.arange(len(mon)); w = .38
for k, (col, c) in enumerate(zip(mon.columns, [C['navy'], C['orange']])):
    b = ax.bar(x + (k - .5) * w, mon[col], w, color=c, label=col[:2])
    for xi, v, d in zip(x + (k - .5) * w, mon[col], days[col].reindex(mon.index)):
        if not np.isfinite(v): ax.text(xi, 0.3, '평시\n없음', ha='center', va='bottom', fontsize=7.5, color=C['ink2']); continue
        ax.text(xi, v + (0.3 if v >= 0 else -0.3), f'{v:+.1f}\n({int(d)}일)', ha='center', va='bottom' if v >= 0 else 'top', fontsize=7.5)
ax.axhline(0, color=C['ink2'], lw=1); ax.set_xticks(x, [f'{m}월' for m in mon.index])
ax.set_ylabel('평시 금액가중 y (%)'); ax.legend(frameon=False)
tot = (E['S'] - E['B']).sum() / E.drop_duplicates(['gkey', '날짜'])['B_하루합'].sum() * 100
ax.set_title(f'A2. 평시 날 월별 편향 (전체 {tot:+.1f}%, 괄호 = 평시 날 수, 특수일 제외)', loc='left', fontsize=11, color=C['ink'])
ax.margins(y=0.25); save(fig, 'A2_월별_편향.png')
TABLES['A2_월별편향'] = mon.round(2).reset_index()


## A3. 이벤트·특수일

In [ ]:
# A3. 이벤트 달력·이벤트 표
EPX = EP2.copy()
dates = pd.date_range(V4['날짜'].min(), V4['날짜'].max())
fig, ax = plt.subplots(figsize=(12, 3.6)); rows = []
for i, reg in enumerate(sorted(V4['대상'].unique())):
    base = 3 * (1 - i)
    for j, (typ, c) in enumerate([('폭염', C['orange']), ('호우', C['blue'])]):
        e = EPX[(EPX['대상'] == reg) & (EPX['종류'] == typ)]
        ax.broken_barh([(mdates.date2num(s), (t - s).days + 1) for s, t in zip(e['시작'], e['종료'])], (base + 1.6 - j * 0.8, 0.6), color=c)
        rows.append((base + 1.9 - j * 0.8, f'{reg[:2]} {typ} ({len(e)}회)'))
    sp = ns2['SPECIAL2_DAYS']; have = set(V4.loc[V4['대상'] == reg, '날짜'])
    for d in sp: ax.broken_barh([(mdates.date2num(d), 1)], (base + 0.0, 0.6), color=C['gray'])
    for d in dates.difference(pd.DatetimeIndex(sorted(have))): ax.broken_barh([(mdates.date2num(d), 1)], (base + 0.0, 0.6), color='#c9ccd3', hatch='///', edgecolor='white')
    rows.append((base + 0.3, f'{reg[:2]} 특수일·제외일'))
ax.set_yticks([r[0] for r in rows], [r[1] for r in rows], fontsize=9)
ax.xaxis.set_major_locator(mdates.MonthLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter('%m월'))
ax.set_xlim(mdates.date2num(dates[0]), mdates.date2num(dates[-1]) + 1); ax.grid(axis='y', visible=False)
ax.set_title('A3. 특보 이벤트(특보가 이어진 기간마다 하나), 특수일(회색: 연휴 ±1일), 제외일(빗금: 한파·대설)', loc='left', fontsize=11, color=C['ink'])
save(fig, 'A3_이벤트_달력.png')
et = EPX.assign(기간=lambda d: d['시작'].dt.strftime('%m/%d') + ' ~ ' + d['종료'].dt.strftime('%m/%d'),
                일수=lambda d: d['날짜'].map(len), 분석일수=lambda d: d['분석일'].map(len))[['대상', '종류', '에피소드', '기간', '일수', '분석일수']]
et['대상'] = et['대상'].str[:2]
TABLES['A3_이벤트'] = table_png(et, 'A3_이벤트표.png', 'A3. 이벤트 목록 (분석일 = 특수일 제외)', col_w=[1.2, 1.2, 1.4, 2.4, 1, 1.2], fs=8.5)


## A4. 취약 판정

In [ ]:
# A4. 전체 판정 분포 · 가짜 사건 점검
ORDER = [('강남(서울동남권)', '폭염'), ('강남(서울동남권)', '호우'), ('춘천', '폭염'), ('춘천', '호우')]
lab = lambda r, t: f'{r[:2]} {t}'
JC = [('취약 후보', C['red']), ('감소 관찰(반복 확인 필요)', C['yellow']), ('구분 어려움', '#c9ccd3'), ('분석범위 보류', C['ink2'])]
X = pd.crosstab([S2['대상'], S2['종류']], S2['전체판정']).reindex(ORDER).fillna(0)
fig, ax = plt.subplots(figsize=(9, 3.6)); left = np.zeros(4); yy = np.arange(4)[::-1]
for k, c in JC:
    v = X[k].to_numpy() if k in X else np.zeros(4)
    ax.barh(yy, v, left=left, color=c, label=k, height=.6)
    for y_, l_, v_ in zip(yy, left, v):
        if v_: ax.text(l_ + v_ / 2, y_, int(v_), ha='center', va='center', fontsize=9, color='white' if c != '#c9ccd3' else C['ink'])
    left += v
ax.set_yticks(yy, [lab(*o) for o in ORDER]); ax.set_xlabel('지역×업종 수'); ax.grid(axis='y', visible=False)
ax.legend(frameon=False, ncol=4, fontsize=8.5, loc='upper center', bbox_to_anchor=(.5, -.18))
ax.set_title(f"A4. 업종별 판정 결과 (전체 {len(S2)} = 152 × 폭염·호우)", loc='left', fontsize=11, color=C['ink'])
save(fig, 'A4_전체판정.png')
TABLES['A4_판정분포'] = X.reset_index()
fake = pd.DataFrame([
    ['폭염 길이 (13~17일)', '19.9%', '12.1%', '10% 초과 → 보조 결과 함께 제시'],
    ['호우 길이 (1~5일)', '9.4%', '6.2%', '기준 이내'],
], columns=['구간', '주 결과 (하루씩 추출)', '보조 (7일 묶음 추출)', '사전 규칙 적용'])
TABLES['A4_가짜사건'] = table_png(fake, 'A4_가짜사건점검.png', 'A4. 가짜 사건 점검: 특보 없는 기간에 같은 판정을 했을 때 "감소" 비율 (기대 5%)',
                                note='반복 해석: 한 이벤트 우연 감소 확률이 20%일 때 3개 중 2개 이상 ≈ 10%, 3개 모두 ≈ 1% → 3/3 이상은 강한 근거. 출처: 3축_재점검 노트북',
                                col_w=[2.6, 2.2, 2.2, 3.4])


## A5. 3축 규칙

In [ ]:
# A5. 3축 계산 규칙 (표)
rule = pd.DataFrame([
    ['Impact (깊이)', '이벤트 중 하루 변화율의 3일 평균 최솟값', '평시 3일 평균 하위 5%보다 얕으면 회복 0일'],
    ['Cumulative (누적)', 'Σ(실제 - 평소) ÷ 평균 하루 평소 매출', '"평소 하루 매출 ○일치"'],
    ['Recovery (회복)', '종료 후 회복 기준 안에 2일 연속 들어온 첫날까지 일수 (최대 14일)', '기준 = -10%와 평시 하위 25% 중 낮은 쪽. 다음 특보로 끊기면 "○일 이상"'],
    ['순위', '같은 지역·기상의 후보끼리 축별 백분위(1 = 가장 취약) 평균', '정보 없는 축은 나머지 축 평균'],
    ['견고성', '한 축 가중치를 50%로 바꿔도 상위 10 중 8~10개 유지', '3축 순위 vs 누적 단독 순위 상관 0.89~0.96 (3축_재점검)'],
], columns=['항목', '계산', '비고'])
TABLES['A5_3축'] = table_png(rule, 'A5_3축규칙.png', 'A5. 3축 계산 규칙', col_w=[2, 5.2, 5.2], fs=8.5)


## A6. 유형 분류

In [ ]:
# A6. 유형 분류 기준 · 업종 성격 × 시간대 유형
crit = pd.DataFrame([
    ['피해 모양', 'Impact 점수·Cumulative 점수 > 0.5', '상대적 심각(둘 다) / 급락(Impact만) / 지속 침체(Cumulative만) / 상대적 완만'],
    ['회복 지연 (꼬리표)', 'Recovery ≥ 3일 ("이상" 포함)', ''],
    ['시간대 차이 작음', '낮·저녁 모두 |변화| < 5%', '낮 = 06~17시, 저녁 = 18~23시'],
    ['시간 이동형', '저녁 증가가 낮 감소폭의 30% 이상', '강남 제과점 1개 (금액 기준으로는 29% → 경계 사례)'],
    ['저녁 집중 감소형', '저녁 감소가 낮 감소의 1.5배 이상', ''],
    ['전일 감소형', '저녁도 낮 감소의 50% 이상 감소', '저녁 영업 없는 업종은 낮 5% 이상 감소면 전일'],
    ['낮 집중 감소형', '그 밖의 낮 감소', ''],
], columns=['유형', '기준 (결과 보기 전에 정함)', '비고'])
TABLES['A6_기준'] = table_png(crit, 'A6_유형기준.png', 'A6. 유형 분류 기준', col_w=[2.4, 4.4, 6], fs=8.5)
kw = pd.DataFrame([(n, ', '.join(k)) for n, k in ns2['NATURE_RULES']], columns=['업종 성격', '키워드 (업종 이름에 포함, 위에서부터)'])
TABLES['A6_키워드'] = table_png(kw, 'A6_업종성격_키워드.png', 'A6. 업종 성격 분류 규칙', col_w=[2.2, 10.5], fs=8)
TT = ['낮 집중 감소형', '전일 감소형', '저녁 집중 감소형', '저녁 감소형', '시간 이동형', '시간대 차이 작음']
H = pd.crosstab(CAND['업종성격'], CAND['시간대유형']).reindex(columns=[t for t in TT if t in set(CAND['시간대유형'])]).fillna(0).astype(int)
H = H.loc[H.sum(axis=1).sort_values(ascending=False).index]
fig, ax = plt.subplots(figsize=(8.5, 4.2))
im = ax.imshow(H.values, cmap='Blues', aspect='auto', vmin=0)
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        v = H.values[i, j]
        if v: ax.text(j, i, v, ha='center', va='center', color='white' if v > H.values.max() / 2 else C['ink'], fontsize=10)
ax.set_xticks(range(H.shape[1]), H.columns, fontsize=9); ax.set_yticks(range(H.shape[0]), H.index, fontsize=9); ax.grid(False)
ax.set_title('A6. 업종 성격 × 시간대 유형 (취약 후보 43개, 4개 지역·기상 합계)', loc='left', fontsize=11, color=C['ink'])
save(fig, 'A6_성격x시간대.png')
TABLES['A6_성격x시간대'] = H.reset_index()


## A7. 취약 후보 43개

In [ ]:
# A7. 취약 후보 43개 전체 표 (지역·기상별 1장씩)
for reg, typ in ORDER:
    t = CAND[(CAND['대상'] == reg) & (CAND['종류'] == typ)].sort_values(['순위', '업종'])
    t = pd.DataFrame({'순위': t['순위'].astype(int), '업종': t['업종'], '반복': t['반복'],
                      'Impact': (t['Impact'] * 100).round(1).map(lambda v: f'{v:.1f}%'),
                      '누적(일치)': t['Cumulative'].round(2), '회복': t['Recovery표시'],
                      '피해 모양': t['피해모양'].replace({'심각형': '상대적 심각형'}), '시간대 유형': t['시간대유형'], '업종 성격': t['업종성격']})
    TABLES[f'A7_{reg[:2]}{typ}'] = table_png(t, f'A7_후보_{reg[:2]}{typ}.png', f'A7. {lab(reg, typ)} 취약 후보 {len(t)}개 (같은 순위 = 공동)',
                                            col_w=[0.8, 3.2, 0.9, 1.1, 1.1, 1.3, 1.7, 1.8, 1.9], fs=8)


## A8. 감소 분해

In [ ]:
# A8. 감소 분해 — 주 결과 vs 여름 객단가 보조, 바뀐 업종, 외지인/거주민
VT = ['방문 감소형', '혼합형', '지출 감소형']
a = pd.crosstab([T['대상'], T['종류']], T['분해_유형']).reindex(ORDER).reindex(columns=VT).fillna(0)
b = pd.crosstab([T['대상'], T['종류']], T['분해_유형_여름']).reindex(ORDER).reindex(columns=VT).fillna(0)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, (M, ttl) in zip(axes, [(a, f'주 결과 (평시 객단가) — 방문 {int(a["방문 감소형"].sum())} · 혼합 {int(a["혼합형"].sum())} · 지출 {int(a["지출 감소형"].sum())}'),
                               (b, f'보조 (여름 객단가) — 방문 {int(b["방문 감소형"].sum())} · 혼합 {int(b["혼합형"].sum())} · 지출 {int(b["지출 감소형"].sum())}')]):
    left = np.zeros(4); yy = np.arange(4)[::-1]
    for k, c in zip(VT, [C['blue'], C['teal'], C['orange']]):
        v = M[k].to_numpy(); ax.barh(yy, v, left=left, color=c, label=k, height=.6)
        for y_, l_, v_ in zip(yy, left, v):
            if v_: ax.text(l_ + v_ / 2, y_, int(v_), ha='center', va='center', color='white', fontsize=9)
        left += v
    ax.set_title(ttl, loc='left', fontsize=10, color=C['ink']); ax.grid(axis='y', visible=False)
axes[0].set_yticks(np.arange(4)[::-1], [lab(*o) for o in ORDER])
axes[0].legend(frameon=False, ncol=3, fontsize=9, loc='upper center', bbox_to_anchor=(1.05, -.12))
fig.suptitle('A8. 방문/지출 분해: 지출 감소형 대부분은 여름 객단가 기준에서 방문 감소로 바뀜 (계절 객단가 영향 가능성)', x=.01, ha='left', fontsize=11, color=C['ink'])
fig.tight_layout(); save(fig, 'A8_방문지출_주vs여름.png')
ch = T[T['분해_유형'] != T['분해_유형_여름']].sort_values(['대상', '종류', '순위'])
ch = pd.DataFrame({'지역·기상': [lab(r, t) for r, t in zip(ch['대상'], ch['종류'])], '순위': ch['순위'].astype(int), '업종': ch['업종'], '본 시간대': ch['분해시간대'],
                   '주 결과': ch['분해_유형'] + ch.apply(lambda r: f" (건수 {r['분해_건수변화']*100:+.0f}%, 객단가 {r['분해_객단가변화']*100:+.0f}%)", axis=1),
                   '여름 기준': ch['분해_유형_여름'] + ch.apply(lambda r: f" (건수 {r['분해_건수변화_여름']*100:+.0f}%, 객단가 {r['분해_객단가변화_여름']*100:+.0f}%)", axis=1)})
TABLES['A8_바뀐업종'] = table_png(ch, 'A8_여름기준_바뀐업종.png', f'A8. 여름 객단가로 보면 유형이 바뀌는 업종 ({len(ch)}개)', col_w=[1.4, 0.8, 3, 1.2, 4.2, 4.2], fs=7.5)
fig, ax = plt.subplots(figsize=(7.5, 5.2))
CU = {'쏠림 없음': C['teal'], '외지인 주도': C['gray'], '거주민 주도': C['blue']}
for k, c in CU.items():
    q = T[T['고객층'] == k]
    ax.scatter(q['거주민변화'] * 100, q['외지인변화'] * 100, s=55, color=c, edgecolor='white', lw=1.2, label=f'{k} {len(q)}', zorder=3)
lim = [min(T['거주민변화'].min(), T['외지인변화'].min()) * 100 - 5, max(T['거주민변화'].max(), T['외지인변화'].max(), 0) * 100 + 5]
ax.plot(lim, lim, color=C['ink2'], ls='--', lw=1); ax.text(lim[1], lim[1], ' 같은 비율로 감소', fontsize=8, color=C['ink2'], va='bottom', ha='right')
for _, r in T[T['고객층'] != '쏠림 없음'].iterrows():
    ax.annotate(f"{r['대상'][:2]}{r['종류']} {r['업종']}", (r['거주민변화'] * 100, r['외지인변화'] * 100), xytext=(5, 4), textcoords='offset points', fontsize=8)
ax.set_xlabel('거주민 매출 변화 (%)'); ax.set_ylabel('외지인 매출 변화 (%)'); ax.legend(frameon=False, fontsize=9, loc='upper left')
ax.set_title('A8. 외지인 vs 거주민 매출 변화 (점선 아래 = 외지인이 더 감소, 강남은 참고용)', loc='left', fontsize=10.5, color=C['ink'])
save(fig, 'A8_외지인_거주민.png')


## A9. 2단계 모델

In [ ]:
# A9. 2단계 모델 — 설정, 변수 중요도, 검증 묶음
m9 = pd.DataFrame([
    ['모델', 'LightGBM, Huber 손실 (α = 0.1) — L2·L1·Huber 중 특보일 MAE 개선율과 업종 순위상관으로 선택'],
    ['하이퍼파라미터', '트리 400 · 학습률 0.03 · 잎 31 · 잎당 최소 200행 · 샘플링 0.8 · 열 0.8 · L2 1.0 · seed 42'],
    ['예측 대상', '시간대 y값 (지역 × 업종 × 시간대 × 날짜), 휴무 칸·특수일 제외'],
    ['변수 14개', '지역·업종·시간대 / 시간대 최고기온·강수·강수시간·습도·풍속 / 일 최고기온·일 강수 / 전3일강수·폭염연속일수 / 폭염·호우 등급'],
    ['검증', '겹치는 이벤트를 묶은 검증 묶음마다 통째로 빼고 학습 (앞뒤 3일도 제외) → 처음 보는 이벤트 예측'],
    ['3축 계산', '예측 y에 실제와 같은 판정·3축 공식 적용. 예측 Recovery는 쓰지 않음(종료 후 경과일 정보 없음)'],
], columns=['항목', '내용'])
TABLES['A9_설정'] = table_png(m9, 'A9_모델설정.png', 'A9. 2단계 예측 모델 설정', col_w=[2, 11], fs=8.5)
imp = (IMP / IMP.sum() * 100).rename(index={'slot': '시간대', '대상': '지역'}).sort_values()
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.barh(imp.index, imp.values, color=C['blue'])
for i, v in enumerate(imp.values): ax.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=8.5)
ax.set_xlabel('gain 비중 (%)'); ax.grid(axis='y', visible=False)
ax.set_title('A9. 변수 중요도 (전체 데이터로 학습한 최종 모델)', loc='left', fontsize=11, color=C['ink'])
save(fig, 'A9_변수중요도.png'); TABLES['A9_변수중요도'] = imp[::-1].round(2).rename('gain %').reset_index()
fig, ax = plt.subplots(figsize=(12, 2.6))
for reg_i, (reg, typ) in enumerate(ORDER):
    e = EP2[(EP2['대상'] == reg) & (EP2['종류'] == typ)]
    ax.broken_barh([(mdates.date2num(s), (t - s).days + 1) for s, t in zip(e['시작'], e['종료'])], (3 - reg_i, 0.6), color=C['orange'] if typ == '폭염' else C['blue'])
for i, c in CL.iterrows():
    ax.axvspan(mdates.date2num(c['시작']), mdates.date2num(c['종료']) + 1, color=C['shade'], zorder=0)
    ax.text(mdates.date2num(c['시작']), 4.1, f'묶음 {i}', fontsize=8, color=C['ink2'])
ax.set_yticks([3.3, 2.3, 1.3, 0.3], [lab(*o) for o in ORDER], fontsize=9); ax.set_ylim(0, 4.6)
ax.xaxis.set_major_locator(mdates.MonthLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter('%m월')); ax.grid(axis='y', visible=False)
ax.set_xlim(mdates.date2num(pd.Timestamp('2025-06-28')), mdates.date2num(pd.Timestamp('2025-10-20')))
ax.set_title(f'A9. 검증 묶음 {len(CL)}개: 묶음마다 그 기간(±3일)을 빼고 학습한 모델로 예측', loc='left', fontsize=11, color=C['ink'])
save(fig, 'A9_검증묶음.png')


## A10. 과거 평균 비교

In [ ]:
# A10. 과거 평균 비교 전체 표 · 후보 찾기
V = SUMM.merge(MAE, on=['대상', '종류']).merge(AXV[AXV['축'] == 'Impact'][['대상', '종류', '모델', '과거 평균']].rename(columns={'모델': 'IM', '과거 평균': 'IP'}), on=['대상', '종류']) \
        .merge(AXV[AXV['축'] == 'Cumulative'][['대상', '종류', '모델', '과거 평균']].rename(columns={'모델': 'CM', '과거 평균': 'CP'}), on=['대상', '종류']).merge(REC, on=['대상', '종류'])
V = V.set_index(['대상', '종류']).loc[ORDER].reset_index()
t10 = pd.DataFrame({'지역·기상': [lab(r, t) for r, t in zip(V['대상'], V['종류'])],
                    '업종 순위상관 (모델/과거)': [f'{a:.2f} / {b:.2f}' for a, b in zip(V['모델_평균'], V['과거평균_평균'])],
                    '모델 승': V['모델승'], '사전 규칙 판단': V['판단'].str.replace('모델이 업종 순위를 더 잘 맞힘', '모델 우세'),
                    'MAE 개선 (모델/과거)': [f'{a*100:+.1f}% / {b*100:+.1f}%' for a, b in zip(V['모델 개선율(평소대로 대비)'], V['과거평균 개선율(평소대로 대비)'])],
                    'Impact 순위상관': [f'{a:.2f} / {b:.2f}' for a, b in zip(V['IM'], V['IP'])],
                    'Cumulative 순위상관': [f'{a:.2f} / {b:.2f}' for a, b in zip(V['CM'], V['CP'])],
                    '후보 재현율': [f'{a:.2f} / {b:.2f}' for a, b in zip(V['모델_재현율'], V['과거 평균_재현율'])],
                    '상위 겹침': [f'{int(a)} / {int(b)} (k={int(k)})' for a, b, k in zip(V['모델_상위겹침'], V['과거 평균_상위겹침'], V['상위k'])]})
TABLES['A10_비교'] = table_png(t10, 'A10_과거평균비교_표.png', 'A10. 기상 모델 vs 과거 평균 예측 (이벤트를 빼고 학습한 예측, MAE 개선은 "평소대로(y=0)" 대비)',
                              note='모든 칸은 "모델 / 과거 평균". 순위·깊이는 모델이 낫고, 후보 목록은 과거 평균이 더 넓게 찾음 → 두 정보를 결합해 운영', col_w=[1.4, 2.4, 1, 1.6, 2.6, 1.8, 2.1, 1.6, 1.9], fs=8.5)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6)); x = np.arange(4); w = .38
for ax, (cm, cp, ttl) in zip(axes, [('모델_재현율', '과거 평균_재현율', '실제 취약 후보 중 예측도 후보로 찾은 비율 (재현율)'), ('모델_정밀도', '과거 평균_정밀도', '예측 후보 중 실제 후보 비율 (정밀도)')]):
    ax.bar(x - w / 2, V[cp], w, color=C['gray'], label='과거 평균 예측'); ax.bar(x + w / 2, V[cm], w, color=C['blue'], label='기상 모델')
    for xi, a_, b_ in zip(x, V[cp], V[cm]):
        ax.text(xi - w / 2, a_ + .02, f'{a_:.2f}', ha='center', fontsize=8); ax.text(xi + w / 2, b_ + .02, f'{b_:.2f}', ha='center', fontsize=8)
    ax.set_xticks(x, [lab(*o) for o in ORDER]); ax.set_ylim(0, 1.15); ax.set_title(ttl, loc='left', fontsize=10, color=C['ink'])
axes[0].legend(frameon=False, fontsize=9)
fig.suptitle('A10. 취약 후보 찾기 — 과거 평균이 더 넓게 찾고, 모델은 더 보수적', x=.01, ha='left', fontsize=11, color=C['ink'])
fig.tight_layout(); save(fig, 'A10_후보찾기.png')


## A11. 재현성 + 저장

In [ ]:
# A11. 재현성 — baseline_GPU_CPU비교 결과가 있으면 표로
hits = sorted(ROOT.rglob('GPU_CPU비교_요약.xlsx'))
if hits:
    bc = pd.read_excel(hits[-1], sheet_name='B비교'); cc = pd.read_excel(hits[-1], sheet_name='후보비교')
    TABLES['A11_B비교'] = table_png(bc, 'A11_GPU_CPU.png', 'A11. baseline 학습 환경별 B 비교 (원래 V4 = Colab T4 GPU)', fs=8)
    cc = cc.fillna('')
    TABLES['A11_후보비교'] = table_png(cc, 'A11_GPU_CPU_후보.png', 'A11. 학습 환경별 취약 후보 (CPU로 학습해도 43개 중 42개 유지)', fs=8)
else:
    rep = pd.DataFrame([
        ['원래 V4 (GPU 1·2회)', '100% 같음', '43개 (17·3·14·9)', '—'],
        ['CPU로 다시 학습', '칸 단위 -9% ~ +10%', '47개 (16·3·15·13)', '42개 겹침, 바뀐 6개는 반복 기준 경계 업종'],
    ], columns=['학습 환경', 'B 차이 (원래 대비)', '취약 후보', '비고'])
    TABLES['A11_재현성'] = table_png(rep, 'A11_GPU_CPU.png', 'A11. 재현성: baseline 학습 환경 민감도', col_w=[2.6, 2.4, 2.4, 4.4])
steps = pd.DataFrame([
    ['01_baseline_V4', '평소 매출 B (GPU 런타임 필수)', 'outputs/38_최종baseline_V4.csv'],
    ['02_취약판정_유형나누기', '판정·3축 순위·유형', 'results/유형나누기/유형나누기_결과.xlsx'],
    ['03_2단계모델', '2단계 모델·과거 평균 비교·예측 순위', 'results/2단계모델/'],
    ['04_감소분해', '방문/지출·외지인/거주민', 'results/감소분해/감소분해_결과.xlsx'],
], columns=['노트북', '하는 일', '결과 파일'])
TABLES['A11_실행순서'] = table_png(steps, 'A11_실행순서.png', 'A11. 재현 코드 실행 순서 (무작위 추출은 모두 시드 고정, xgboost 3.4.1)', col_w=[3, 3.6, 4.6])
with pd.ExcelWriter(OUT / '부록표.xlsx') as xw:
    for k, v in TABLES.items(): v.to_excel(xw, sheet_name=k[:31], index=False)
print('저장:', OUT / '부록표.xlsx'); print('그림 수:', len(list(OUT.glob('*.png'))))
